In [6]:
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import Dataset

In [7]:
train_df = pd.read_csv("../data_organizing/train_set.csv", dtype={"nmbr": str})
test_df = pd.read_csv("../data_organizing/test_set.csv", dtype={"nmbr": str})

feature_columns = ["area", "ll", "nn"]
target_column = "zurich_class"

X_train = train_df[feature_columns]
y_train = train_df[target_column]

X_test = test_df[feature_columns]
y_test = test_df[target_column]

In [ ]:
#CUSTOM DATASET (HEAVY ASSIST FROM CLAUDE)
class SunspotDataset(Dataset):
    def __init__(self, dataframe, crops_folder, class_to_idx):
        self.data = dataframe[dataframe["crop_filename"].notna()].reset_index(drop=True)
        self.crops_folder = Path(crops_folder)
        self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        crop_path = self.crops_folder / row["crop_filename"]
        image = np.load(crop_path).astype("float32")

        image = torch.from_numpy(image).unsqueeze(0)

        label = self.class_to_idx[row["zurich_class"]]

        return image, label


classes = sorted(train_df["zurich_class"].unique())
class_to_idx = {cls: i for i, cls in enumerate(classes)}
print(class_to_idx)

crops_folder = r"D:\IC-DeepLearning\cropeed_data"

train_dataset = SunspotDataset(train_df, crops_folder, class_to_idx)
test_dataset = SunspotDataset(test_df, crops_folder, class_to_idx)

print("Train example:", len(train_dataset))
print("Test example:", len(test_dataset))

image, label = train_dataset[0]
print("Image format:", image.shape, "| Label:", label)

{'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'H': 6}
Train example: 8190
Test example: 941
Image format: torch.Size([1, 101, 101]) | Label: 4
